# File Connection Checks

Checks that file-level identifiers referenced across GTFS tables actually exist where expected.

In [1]:
from pathlib import Path
import sys

_current = Path.cwd().resolve()
for _candidate in [_current, *_current.parents]:
    if (_candidate / "data_validation" / "checks" / "commons.py").exists():
        _project_root = _candidate
        break
else:
    raise FileNotFoundError("data_validation/checks/commons.py not found.")

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from data_validation.checks.commons import (
    PATHWAYS_FILE,
    ROUTES_FILE,
    STOPS_FILE,
    STOP_TIMES_FILE,
    TRIPS_FILE,
    check_missing_files,
    load_from_stop_ids,
    load_route_ids,
    load_stop_ids,
    load_to_stop_ids,
    load_trip_ids,
)

In [3]:
def main() -> None:
    check_missing_files([PATHWAYS_FILE, STOP_TIMES_FILE, STOPS_FILE, TRIPS_FILE, ROUTES_FILE])

    pathways_from_stop_ids = load_from_stop_ids(PATHWAYS_FILE)
    pathways_to_stop_ids = load_to_stop_ids(PATHWAYS_FILE)
    stop_times_stop_ids = load_stop_ids(STOP_TIMES_FILE)
    stops_stop_ids = load_stop_ids(STOPS_FILE)

    missing_pathways_from = sorted(ids for ids in pathways_from_stop_ids if ids not in stops_stop_ids)
    missing_pathways_to = sorted(ids for ids in pathways_to_stop_ids if ids not in stops_stop_ids)
    missing_stop_times = sorted(ids for ids in stop_times_stop_ids if ids not in stops_stop_ids)

    print(f"----- All stop_id from pathways.txt and stop_times.txt exist in stops.txt? -----")
    if not missing_pathways_from:
        print(" - All correct: no from_stop_id from pathways.txt is missing in stops.txt")
    else:
        print(f" - MISSING {len(missing_pathways_from)} from_stop_id from pathways.txt in stops.txt:")
        for sid in missing_pathways_from:
            print("   -", sid)

    if not missing_pathways_to:
        print(" - All correct: no to_stop_id from pathways.txt is missing in stops.txt")
    else:
        print(f" - MISSING {len(missing_pathways_to)} to_stop_id from pathways.txt in stops.txt:")
        for sid in missing_pathways_to:
            print("   -", sid)

    if not missing_stop_times:
        print(" - All correct: no stop_id from stop_times.txt is missing in stops.txt")
    else:
        print(f" - MISSING {len(missing_stop_times)} stop_id from stop_times.txt in stops.txt:")
        for sid in missing_stop_times:
            print("   -", sid)

    trips_route_ids = load_route_ids(TRIPS_FILE)
    routes_route_ids = load_route_ids(ROUTES_FILE)
    missing_route_ids = sorted(ids for ids in trips_route_ids if ids not in routes_route_ids)

    print(f"Unique route_id in trips: {len(trips_route_ids)}")
    print(f"Unique route_id in routes: {len(routes_route_ids)}")

    print(f"\n----- All route_id from trips.txt exist in routes.txt? -----")
    if not missing_route_ids:
        print("All correct: all route_id present in trips also appear in routes.")
    else:
        print(f"MISSING {len(missing_route_ids)} route_id (present in trips but not in routes):")
        for rid in missing_route_ids:
            print(f"- {rid}")

    stop_times_trip_ids = load_trip_ids(STOP_TIMES_FILE)
    trips_trip_ids = load_trip_ids(TRIPS_FILE)
    missing_trip_ids = sorted(ids for ids in stop_times_trip_ids if ids not in trips_trip_ids)

    print(f"Unique trip_id in stop_times: {len(stop_times_trip_ids)}")
    print(f"Unique trip_id in trips: {len(trips_trip_ids)}")

    print(f"\n----- All trip_id from stop_times.txt exist in trips.txt? -----")
    if not missing_trip_ids:
        print("All correct: all trip_id present in stop_times also appear in trips.")
    else:
        print(f"MISSING {len(missing_trip_ids)} trip_id (present in stop_times but not in trips):")
        for tid in missing_trip_ids:
            print(f"- {tid}")

main()

----- All stop_id from pathways.txt and stop_times.txt exist in stops.txt? -----
 - All correct: no from_stop_id from pathways.txt is missing in stops.txt
 - All correct: no to_stop_id from pathways.txt is missing in stops.txt
 - All correct: no stop_id from stop_times.txt is missing in stops.txt
Unique route_id in trips: 114
Unique route_id in routes: 114

----- All route_id from trips.txt exist in routes.txt? -----
All correct: all route_id present in trips also appear in routes.
Unique trip_id in stop_times: 48987
Unique trip_id in trips: 48987

----- All trip_id from stop_times.txt exist in trips.txt? -----
All correct: all trip_id present in stop_times also appear in trips.
